# Análisis Comparativo de Modelos de NLP para Análisis de Sentimientos
**Rama de IA:** Procesamiento de Lenguaje Natural (NLP)
**Caso de uso:** Clasificación de reseñas de texto (Positivo/Negativo) utilizando modelos preentrenados de Hugging Face.

In [20]:
# Verificación de la GPU asignada por Google Colab
!nvidia-smi

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Utilizando dispositivo: {device}")

Sun Oct 11 22:08:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P0             31W /   70W |    1483MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# Instalación de librerías necesarias
!pip install -q transformers datasets evaluate scikit-learn tabulate huggingface_hub --quiet

In [21]:
from huggingface_hub import login
from google.colab import userdata

# Reemplaza 'TU_TOKEN_AQUI' con tu token de acceso de Hugging Face.
# Asegúrate de que el token tenga permisos de lectura (Read).
hf_token = userdata.get('hug_face_token')

login(token=hf_token)
print("Autenticación completada exitosamente.")

Autenticación completada exitosamente.


In [22]:
from datasets import load_dataset
import pandas as pd

# Cargar dataset de IMDB (reduciremos el tamaño para que la evaluación sea rápida en Colab)
print("Cargando dataset IMDB...")
dataset = load_dataset("stanfordnlp/imdb", split="test")

# Tomamos una muestra aleatoria y balanceada de 1000 registros para la prueba
sample_dataset = dataset.shuffle(seed=42).select(range(1000))
textos = sample_dataset['text']
etiquetas_reales = sample_dataset['label'] # 0: Negativo, 1: Positivo

print(f"Dataset cargado. Total de muestras para evaluación: {len(textos)}")

# Definición de los tres modelos a comparar
modelos_hf = {
    "DistilBERT": "distilbert-base-uncased-finetuned-sst-2-english",
    "BERT-base": "textattack/bert-base-uncased-imdb",
    "RoBERTa": "aychang/roberta-base-imdb"
}

Cargando dataset IMDB...
Dataset cargado. Total de muestras para evaluación: 1000


In [16]:
from transformers import pipeline
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tabulate import tabulate

def evaluar_modelo(nombre_modelo, model_id, textos, y_true):
    print(f"\n--- Evaluando {nombre_modelo} ({model_id}) ---")

    # Inicializar el pipeline en la GPU si está disponible
    clasificador = pipeline("sentiment-analysis", model=model_id, device=0 if device=="cuda" else -1)

    # Medir latencia
    inicio = time.time()
    predicciones_crudas = clasificador(textos, truncation=True, max_length=512, batch_size=16)
    fin = time.time()

    latencia_total = fin - inicio
    latencia_promedio = (latencia_total / len(textos)) * 1000 # en milisegundos

    # Mapear predicciones a 0 y 1 (depende del modelo, algunos devuelven 'POSITIVE'/'NEGATIVE', otros 'LABEL_1'/'LABEL_0')
    y_pred = []
    for p in predicciones_crudas:
        label = p['label'].lower()
        if 'pos' in label or '1' in label:
            y_pred.append(1)
        else:
            y_pred.append(0)

    # Calcular métricas
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='binary')
    rec = recall_score(y_true, y_pred, average='binary')
    f1 = f1_score(y_true, y_pred, average='binary')

    return {
        "Modelo": nombre_modelo,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "Latencia (ms/seq)": round(latencia_promedio, 2)
    }

# Diccionario para guardar resultados
resultados = []

In [18]:
import warnings
warnings.filterwarnings("ignore")

# Forzar la conversión a listas estándar de Python
textos_lista = list(textos)
etiquetas_lista = list(etiquetas_reales)

# Ejecutar la evaluación para cada modelo
for nombre, model_id in modelos_hf.items():
    resultado = evaluar_modelo(nombre, model_id, textos_lista, etiquetas_lista)
    resultados.append(resultado)


--- Evaluando DistilBERT (distilbert-base-uncased-finetuned-sst-2-english) ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


--- Evaluando BERT-base (textattack/bert-base-uncased-imdb) ---


config.json:   0%|          | 0.00/511 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [19]:
# Imprimir tabla comparativa
print("\n=== TABLA COMPARATIVA DE RESULTADOS ===")
headers = resultados[0].keys()
tabla = [list(r.values()) for r in resultados]
print(tabulate(tabla, headers=headers, tablefmt="grid"))


=== TABLA COMPARATIVA DE RESULTADOS ===
+------------+------------+-------------+----------+------------+---------------------+
| Modelo     |   Accuracy |   Precision |   Recall |   F1-Score |   Latencia (ms/seq) |
+============+============+=============+==========+============+=====================+
| DistilBERT |      0.881 |      0.8917 |   0.8607 |     0.8759 |               17.43 |
+------------+------------+-------------+----------+------------+---------------------+
| BERT-base  |      0.928 |      0.9426 |   0.9078 |     0.9248 |               30.5  |
+------------+------------+-------------+----------+------------+---------------------+
| RoBERTa    |      0.94  |      0.928  |   0.9508 |     0.9393 |               30.33 |
+------------+------------+-------------+----------+------------+---------------------+


### Conclusiones Parciales del Cuaderno
1. **Rendimiento Computacional (Latencia):** Se observa que **DistilBERT** ofrece la latencia por secuencia más baja, justificando su arquitectura destilada, lo cual es ideal para entornos de producción con restricciones de tiempo de respuesta.
2. **Desempeño de Clasificación (Métricas):** Modelos más robustos como **BERT-base** o **RoBERTa** tienden a mostrar mejores resultados en F1-Score y Accuracy, ya que capturan mejor el contexto bidireccional y las sutilezas lingüísticas de reseñas largas.
3. **Trade-off:** La elección final dependerá del caso de negocio. Si el sistema de reseñas requiere procesamiento en tiempo real masivo, el sacrificio mínimo en Accuracy de DistilBERT compensa su velocidad. Si es un procesamiento batch analítico, RoBERTa es superior.